# Lab 6: Spark on your computer

Run the cells one at a time. The flights are in `../shared/output/flights`, and the web UI of Spark is at <http://localhost:4040> while the session is open.

## Start Spark

In [ ]:
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder.master('local[*]').appName('lab06').config(
    'spark.driver.memory', '4g').config('spark.sql.adaptive.enabled',
                                        'false').getOrCreate())
spark.sparkContext.defaultParallelism

## Read the flights

The schema is given explicitly.

In [ ]:
schema = (
    'flight_id long, flight_date date, airline_code string, '
    'flight_number string, tail_number string, origin string, '
    'destination string, scheduled_departure timestamp, '
    'actual_departure timestamp, distance_km int, cancelled int')
flights = spark.read.csv('../shared/output/flights',
                         header=True,
                         schema=schema)
flights.printSchema()
flights.rdd.getNumPartitions()

## An action: count the flights

In [ ]:
flights.count()

## Transformations are lazy

These lines only note the plan. Open the Jobs tab of the web UI: no new job appears.

In [ ]:
flown = (flights.filter(F.col('cancelled') == 0).withColumn(
    'delay_min', (F.col('actual_departure').cast('long') -
                  F.col('scheduled_departure').cast('long')) / 60))
late = flown.filter(F.col('delay_min') > 15)

An action runs the whole plan.

In [ ]:
late.count()

## Partitions, stages and the shuffle

Grouping by origin needs a shuffle. The plan shows it as an `Exchange`.

In [ ]:
by_origin = flights.groupBy('origin').count()
by_origin.explain()
by_origin.orderBy(F.desc('count')).show(5)

## The same query in Spark SQL

In [ ]:
flights.createOrReplaceTempView('flights')
by_origin_sql = spark.sql(
    'SELECT origin, count(*) AS count FROM flights GROUP BY origin')
by_origin_sql.explain()
by_origin_sql.orderBy(F.desc('count')).show(5)

## A join with a small table

The airports table is small, so Spark can send a copy of it to every executor instead of moving the flights.

In [ ]:
airports = spark.read.csv('../shared/output/airports',
                          header=True,
                          inferSchema=True)
by_country = (flights.join(
    airports, flights.origin == airports.airport_code).groupBy(
        'country').count().orderBy(F.desc('count')))
by_country.explain()
by_country.show()

## A window function: running total of delay minutes per airline

In [ ]:
daily = flown.groupBy('airline_code', 'flight_date').agg(
    F.sum('delay_min').alias('delay_min_total'))
running = Window.partitionBy('airline_code').orderBy('flight_date')
total = daily.withColumn('running_delay_min',
                         F.sum('delay_min_total').over(running))
total.filter(F.col('airline_code') == 'AND').orderBy('flight_date').show(5)

## Spark and pandas on the same calculation

Both read all the files and compute the average delay of each airline.

In [ ]:
import time

start = time.time()
spark_result = (flown.groupBy('airline_code').agg(
    F.avg('delay_min').alias('avg_delay_min')).orderBy(
        'airline_code').toPandas())
print(f'Spark: {time.time() - start:.1f} s')
spark_result

In [ ]:
import glob

import pandas as pd

start = time.time()
columns = [
    'airline_code', 'scheduled_departure', 'actual_departure', 'cancelled'
]
frames = [
    pd.read_csv(path,
                usecols=columns,
                parse_dates=['scheduled_departure', 'actual_departure'])
    for path in sorted(glob.glob('../shared/output/flights/*.csv'))
]
data = pd.concat(frames)
data = data[data['cancelled'] == 0]
data['delay_min'] = (data['actual_departure'] -
                     data['scheduled_departure']).dt.total_seconds() / 60
pandas_result = data.groupby('airline_code')['delay_min'].mean()
print(f'pandas: {time.time() - start:.1f} s')
pandas_result

## Stop Spark

In [ ]:
spark.stop()